In [ ]:
import os
from google.colab import drive, userdata

# Secrets do Colab (ícone da chave, à esquerda): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
from getpass import getpass

for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    try:
        os.environ[k] = userdata.get(k)          # Secrets do Colab (no browser)
    except Exception:
        os.environ[k] = getpass(f"{k}: ")        # noutro sítio: pede o valor sem o mostrar
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic

from google.colab import drive, userdata

drive.mount("/content/drive")          # os checkpoints vão para o Drive: sobrevivem a desconexões
RUN_DIR = "/content/drive/MyDrive/wiki-bio-distill/run-v1"

!git clone https://github.com/pinthoz/wiki-bio-distill.git /content/repo
%cd /content/repo
!mkdir -p data preds
!for f in train val test; do aws s3 cp s3://$DATA_BUCKET/v1/$f.jsonl data/; done

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/train.jsonl to data/train.jsonl
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/val.jsonl to data/val.jsonl
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/test.jsonl to data/test.jsonl


In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from common import SYSTEM_TEACHER
from gen import generate_preds

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")

test = [json.loads(l) for l in open("data/test.jsonl")]
# O base recebe o prompt detalhado: é o melhor que consegue sem treino
generate_preds(base, tokenizer, test, SYSTEM_TEACHER, "preds/base.jsonl")
!python evaluate.py preds/base.jsonl

SyntaxError: invalid syntax (4191043622.py, line 3)